In [12]:
import pickle
import polars as pl
from lxml import etree

AXIELL_MARC_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-marc-works.parquet"
AXIELL_RAW_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-raw-works.parquet"

def save_parquet_snapshot(data: dict, path: str):
    df = pl.DataFrame({
        "id": list(data.keys()),
        "body": [pickle.dumps(v) for v in data.values()],
    })
    df.write_parquet(path)


def from_parquet_snapshot(path: str):
    df = pl.read_parquet(path)
    return {row["id"]: pickle.loads(row["body"]) for row in df.iter_rows(named=True)}


def axiell_raw_from_parquet_snapshot():
    data = from_parquet_snapshot(AXIELL_RAW_WORKS_PATH)
    return {k: etree.fromstring(v) for k, v in data.items()}    

In [17]:
from pathlib import Path
from typing import Any
import lxml.etree as ET
from utils.marc import parse_single_marc_record

STYLESHEET_PATH = "/Users/brychtas/Documents/GitHub/axiell-collections-xslt/data/stylesheets/axc_to_marcxml_collect.xsl"

PARSER = ET.XMLParser(remove_blank_text=True)
XSLT_DOC = ET.parse(STYLESHEET_PATH, PARSER)
TRANSFORM = ET.XSLT(XSLT_DOC)

def prepare_for_xslt(root: Any) -> ET._Element:
    """Wrap a raw Axiell record tree in the adlibXML/recordList structure the
    XSLT expects, stripping the OAI-PMH default namespace from all elements."""
    # Strip the OAI default namespace so XSLT template names match
    for elem in root.iter():
        elem.tag = ET.QName(elem).localname
    adlib = ET.Element("adlibXML")
    record_list = ET.SubElement(adlib, "recordList")
    record_list.append(root)
    return adlib


def apply_xslt(
    xml_doc: Any,
    **xslt_params: Any,
):
    formatted_params = {
        key: ET.XSLT.strparam(str(value)) for key, value in xslt_params.items()
    }

    return TRANSFORM(xml_doc, **formatted_params)


In [13]:
axiell_raw_works = axiell_raw_from_parquet_snapshot()
len(raw_axiell_works)

227789


In [18]:
from io import BytesIO
from pymarc import XMLWriter, parse_xml_to_array

axiell_marc_works = {}

for i, (work_id, raw_record) in enumerate(axiell_raw_works.items()):
    # Skip Mimsy works
    if int(work_id.split(":")[1]) < 20000:
        continue

    xslt_result = apply_xslt(prepare_for_xslt(raw_record), control_003="UkLW")
    record = parse_xml_to_array(BytesIO(bytes(xslt_result)))[0]
    
    axiell_marc_works[work_id] = record
    
    if i % 10000 == 0:
        print(i)

20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000


In [24]:
save_parquet_snapshot(axiell_marc_works, AXIELL_MARC_WORKS_PATH)